# Exercise 08 — Text Generation with Hugging Face

In week 6 you sampled text from your mini GPT, and you called GPT-2's `generate` once with greedy decoding and once with nucleus sampling. Last week you used `generate` to let Qwen solve math problems. This week is about that one method: how a language model, which only ever predicts *the next token*, is turned into something that writes whole texts, and how the choices made while doing so change what comes out.

This week has three notebooks:

- **The `generate` API** (this notebook): the knobs that Hugging Face gives you, and what `generate` actually returns.
- **Decoding strategies** (notebook 1): greedy search, beam search and sampling with temperature, top-k and top-p, compared with numbers and not only by eye.
- **Decoding from scratch** (notebook 2): you implement these strategies yourself, check them against Hugging Face, and plug your own rules into `generate`.

This notebook is less about writing algorithms and more about getting to know the tool. Most tasks are short; take the time to play with the settings.

## What you will do
1. **`generate`**: what goes in, what comes out, and how to get only the new text.
2. **`GenerationConfig`, length and stopping**: when does generation end?
3. **Shaping the output**: repetition penalties, n-gram blocking and banned words.
4. **Looking inside**: the scores of every step, token probabilities, and what the logits processors do.
5. **Batched generation**: why decoder-only models need *left* padding.
6. **Multiple-choice questions**: to check your understanding.

## How to work through it
- Run the cells **in order** and fill in **every `# TODO`**.
- Tasks are numbered (**1.1**, **1.2**, …). Tasks that say *Your answer here* want a short written answer, not code.
- Most implementation tasks are followed by a **✅ Check** cell that verifies your work automatically. Run it and make sure it passes before you move on.

> **Heads up: this notebook downloads GPT-2** (~550 MB) unless it is still cached from week 6. Everything runs on a CPU in a few seconds per cell.

In [23]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig, set_seed

torch.manual_seed(0)

# Use the GPU if there is one; a CPU is enough for this notebook.
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using mps device


## 1. `generate`: what goes in and what comes out

We use **GPT-2** (124 M parameters), the same model as in week 6. It is small enough for a CPU and, being a plain pretrained model, it shows the strengths and the failure modes of each decoding method very clearly.

GPT-2 has no padding token. `generate` needs one (for example to fill up finished sequences in a batch), so we tell it to use the end-of-text token `<|endoftext|>` for padding. Setting it once in the model's generation config saves us from passing `pad_token_id=...` to every call, and from a warning each time.

In [24]:
MODEL_NAME = "openai-community/gpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

model.generation_config.pad_token_id = tokenizer.eos_token_id
print(f"{sum(p.numel() for p in model.parameters()):,} parameters, vocabulary of {len(tokenizer):,} tokens")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

124,439,808 parameters, vocabulary of 50,257 tokens


`generate` takes the tokenized prompt (the `input_ids` and the `attention_mask`) and returns a tensor of token ids of shape `(batch_size, prompt_length + new_tokens)`. Note that the output **starts with the prompt**: a decoder-only model simply keeps appending to its input.

In [25]:
inputs = tokenizer("Once upon a time", return_tensors="pt").to(device)
print("input ids: ", inputs["input_ids"])

outputs = model.generate(**inputs, max_new_tokens=30)
print("output ids:", outputs)
print()
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

input ids:  tensor([[7454, 2402,  257,  640]], device='mps:0')
output ids: tensor([[7454, 2402,  257,  640,   11,  262,  995,  373,  257, 1295,  286, 1049,
         8737,  290, 1049, 3514,   13,  383,  995,  373,  257, 1295,  286, 1049,
         3514,   11,  290,  262,  995,  373,  257, 1295,  286, 1049]],
       device='mps:0')

Once upon a time, the world was a place of great beauty and great danger. The world was a place of great danger, and the world was a place of great


**1.1 Complete `continue_text`**, which returns only the *new* text, without the prompt. All keyword arguments are passed on to `generate`, so you can use it with any setting later on.

Hint: the prompt has `inputs["input_ids"].shape[1]` tokens. Slice them off the output *before* decoding. (Slicing the decoded string by `len(prompt)` characters is fragile: decoding does not always give back exactly the characters you put in.)

In [26]:
def continue_text(prompt, **generate_kwargs):
    # Generate a continuation of the prompt and return only the new text
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, **generate_kwargs)
    new_ids = outputs[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)


print(repr(continue_text("Once upon a time", max_new_tokens=30)))

', the world was a place of great beauty and great danger. The world was a place of great danger, and the world was a place of great'


In [27]:
# ✅ Check continue_text
text = continue_text("Once upon a time", max_new_tokens=10)
assert isinstance(text, str), "continue_text should return a string"
assert not text.startswith("Once upon a time"), "the prompt should not be part of the returned text"
assert len(tokenizer(text)["input_ids"]) <= 12, "max_new_tokens=10 should give (about) 10 new tokens"
assert text == continue_text("Once upon a time", max_new_tokens=10), "by default generate is deterministic (greedy)"
print("Looks good ✅")

Looks good ✅


**1.2 Read the continuation of *Once upon a time* above.** What is wrong with it? Without any settings, `generate` uses **greedy decoding**: at every step it takes the single most likely next token. Why would that lead to this kind of text? (You saw the same thing in week 6.)

---

*Answer:*

The text is fluent at first but soon starts to **repeat itself**: the same phrase or sentence comes back again and again (with GPT-2, "Once upon a time" typically turns into a loop about war or the United States). It also does not read like a fairy tale at all.

Greedy decoding always takes the most likely token. Once a phrase has appeared in the context, repeating it becomes *more* likely (copying from the context is one of the easiest things a Transformer learns), so the most likely continuation of a repeated phrase is the same phrase again, and the model gets stuck in a loop it can never leave. Greedy decoding also has no way to choose a less likely word that would lead to a better text later on.

---

## 2. `GenerationConfig`: length and stopping

Every argument you can pass to `generate` is a field of a [`GenerationConfig`](https://huggingface.co/docs/transformers/main_classes/text_generation#transformers.GenerationConfig). The model comes with a default one, which `generate` uses for every argument you do not pass. For GPT-2 it is almost empty, which means greedy decoding:

In [28]:
print(model.generation_config)

GenerationConfig {
  "bos_token_id": 50256,
  "eos_token_id": 50256,
  "pad_token_id": 50256
}



Instead of passing many keyword arguments to every call, you can also collect them in your own `GenerationConfig` and pass it with `generation_config=...`. This is handy when you want to reuse the same settings, or save them together with a model. The two calls below do the same thing:

In [29]:
config = GenerationConfig(max_new_tokens=20, do_sample=False, pad_token_id=tokenizer.eos_token_id)

a = model.generate(**inputs, generation_config=config)
b = model.generate(**inputs, max_new_tokens=20, do_sample=False)
print(torch.equal(a, b))

True


### When does generation stop?

Generation stops as soon as **one** of these happens:

- **`max_new_tokens`** new tokens have been generated (always set it, because a model may never stop on its own),
- the model generates the **end-of-sequence token** (`eos_token_id`). For GPT-2 that is `<|endoftext|>`, which separated documents during pretraining. Chat models have their own end-of-turn tokens,
- one of the **`stop_strings`** appears in the generated text (`generate` then also needs the `tokenizer`, to check strings rather than token ids),
- **`max_time`** seconds have passed.

**`min_new_tokens`** works the other way round: before that many tokens have been generated, the EOS token is forbidden (its probability is set to zero).

**2.1 Complete `first_sentence`**, which generates only the first sentence of a continuation: generation stops at the first `"."`, `"!"` or `"?"`.

In [30]:
def first_sentence(prompt, max_new_tokens=60):
    # The continuation of the prompt up to and including the first ".", "!" or "?"
    return continue_text(
        prompt,
        max_new_tokens=max_new_tokens,
        stop_strings=[".", "!", "?"],
        tokenizer=tokenizer,
    )


for prompt in ["The weather in Aarhus is", "My favourite food is", "The meeting was cancelled because"]:
    print(f"{prompt}|{first_sentence(prompt)}")

The weather in Aarhus is| not as bad as it was in the past.
My favourite food is| the chicken.
The meeting was cancelled because| of the "unacceptable" nature of the meeting.


In [31]:
# ✅ Check first_sentence
for prompt in ["The weather in Aarhus is", "My favourite food is", "The meeting was cancelled because"]:
    sentence = first_sentence(prompt)
    n_tokens = len(tokenizer(sentence)["input_ids"])
    assert sentence.endswith((".", "!", "?")) or n_tokens >= 58, f"generation should stop at an end of sentence: {sentence!r}. Did you pass the stop strings and the tokenizer?"
print("Looks good ✅")

Looks good ✅


### The EOS token is just a token

To the model, the end-of-sequence token is a token like any other: generation stops because `generate` *checks* for it after every step. That means you can declare any token to be the EOS token. For example, if you make the full stop `"."` the EOS token, generation stops at the end of the first sentence, as above, but without checking strings.

**2.2 Generate a continuation that is at least 20 tokens long and ends at a full stop.** Use `eos_token_id` and `min_new_tokens`.

Hint: `tokenizer.convert_tokens_to_ids(".")` gives you the id of the full stop.

In [32]:
period_id = tokenizer.convert_tokens_to_ids(".")
prompt = "The weather in Aarhus is"

inputs = tokenizer(prompt, return_tensors="pt").to(device)
outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    eos_token_id=period_id,
    min_new_tokens=20,
)
new_ids = outputs[0, inputs["input_ids"].shape[1]:]
print(f"{len(new_ids)} new tokens: {prompt}|{tokenizer.decode(new_ids)}")

28 new tokens: The weather in Aarhus is| not as bad as it was in the past, but it is still a bit of a challenge to get to the top of the mountain.


In [33]:
# ✅ Check your settings
ids = new_ids.tolist()
assert period_id not in ids[:19], "the full stop should be forbidden during the first 20 new tokens. Did you set min_new_tokens?"
assert ids[-1] == period_id or len(ids) == 80, "generation should end at the first full stop. Did you set eos_token_id?"
print("Looks good ✅")

Looks good ✅


**2.3 Stopping in practice.** A chat model is asked a question and should answer it and then stop. How does it know when to stop? And what would happen if you prompted the *base* GPT-2 with `"Question: What is the capital of Denmark?\nAnswer:"` and only set `max_new_tokens=200`?

---

*Answer:*

A chat model has been fine-tuned (week 7) on conversations in which every assistant reply ends with a special end-of-turn token. It learns to produce that token when its answer is complete, and `generate` stops there because it is the model's EOS token (or one of several).

The base GPT-2 has never been trained to stop after an answer. It continues the *document*: after the answer it will typically write another question, another answer, a different topic, ... until the 200 tokens are used up. It only stops early if it happens to produce `<|endoftext|>`, which it does rarely. That is why you need `max_new_tokens`, or stop strings such as `"\nQuestion:"`, when you use a base model for question answering.

---

## 3. Shaping the output

Some arguments of `generate` do not change *how* the next token is chosen, but *which tokens are allowed* and how likely they are. Before each choice, `generate` passes the logits through a list of **logits processors**, each of which modifies them. Three of them:

- **`repetition_penalty`** (Keskar et al., 2019): makes every token that already appears in the text less likely. A value of 1.0 means no penalty; 1.2–1.3 is common. You implement it yourself in notebook 2.
- **`no_repeat_ngram_size`**: forbids any n-gram (sequence of *n* tokens) that has already been generated. With `no_repeat_ngram_size=3`, no three tokens in a row can occur twice.
- **`bad_words_ids`**: a list of token sequences that may never be generated.

The helper below finds n-grams of token ids that occur more than once.

In [34]:
def repeated_ngrams(ids, n):
    # The n-grams (as tuples of token ids) that occur more than once in ids
    ngrams = [tuple(ids[i : i + n]) for i in range(len(ids) - n + 1)]
    return {ngram for ngram in ngrams if ngrams.count(ngram) > 1}


def generate_ids(prompt, **generate_kwargs):
    # The new token ids of a continuation of the prompt, as a list
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, **generate_kwargs)
    return outputs[0, inputs["input_ids"].shape[1]:].tolist()


prompt = "I enjoy walking with my cute dog"
greedy_ids = generate_ids(prompt, max_new_tokens=60)
print(tokenizer.decode(greedy_ids))
print(f"\n{len(repeated_ngrams(greedy_ids, 3))} different 3-grams occur more than once")

, but I'm not sure if I'll ever be able to walk with my dog. I'm not sure if I'll ever be able to walk with my dog.

I'm not sure if I'll ever be able to walk with my dog. I'm not sure if I'll ever

16 different 3-grams occur more than once


**3.1 Fix the repetition.** Generate a greedy continuation of the same prompt with a repetition penalty of 1.3 (`rep_ids`), and one in which no 3-gram is repeated (`ngram_ids`).

In [35]:
rep_ids = generate_ids(prompt, max_new_tokens=60, repetition_penalty=1.3)
ngram_ids = generate_ids(prompt, max_new_tokens=60, no_repeat_ngram_size=3)

print("===== repetition_penalty=1.3 =====")
print(tokenizer.decode(rep_ids))
print("\n===== no_repeat_ngram_size=3 =====")
print(tokenizer.decode(ngram_ids))

===== repetition_penalty=1.3 =====
, but I'm not sure if it's because of the dogs or just a lack.
The first time we met was in our neighborhood on an afternoon when there were no cars around and everyone had their own car parked outside to drive home from work (we didn't have any). We walked up

===== no_repeat_ngram_size=3 =====
, but I'm not sure if I'll ever be able to walk with my dog. I'm a little bit of a dog lover, but it's not like I'm going to be able walk with a dog.

I'm not a big fan of the idea of having a dog,


In [36]:
# ✅ Check your generations
assert len(rep_ids) == 60 and len(ngram_ids) == 60, "generate 60 new tokens"
assert rep_ids != greedy_ids, "the repetition penalty should change the output"
# the generated 3-grams must not repeat, also not a 3-gram that overlaps the prompt
full = tokenizer(prompt)["input_ids"] + ngram_ids
assert not repeated_ngrams(full, 3), "no 3-gram should occur twice. Did you set no_repeat_ngram_size=3?"
print("Looks good ✅")

Looks good ✅


**3.2 Look at the two continuations.** Is the repetition gone? Did the texts get *good*? Can you think of a text in which forbidding all repeated 3-grams would be a problem?

---

*Answer:*

Both remove the obvious loop, but the texts are not necessarily good: they tend to wander, and the n-gram block sometimes forces odd word choices, because the model has to avoid a phrase it "wants" to use. The repetition penalty also punishes *single* tokens that occurred before, including words like "the" or "my dog", so with a large penalty the text starts avoiding normal words.

N-gram blocking is a blunt instrument for texts in which repeated phrases are *correct*: an article about **New York** can only mention "New York" once with `no_repeat_ngram_size=2`; a recipe repeats "1 cup of"; code repeats variable names and patterns like `for i in range(`; a legal text repeats defined terms; a song repeats its chorus. It treats the symptom (repetition) rather than the cause (greedy and beam search favour the most likely, safest continuation), which is what sampling, in notebook 1, addresses.

---

### Banning words

`bad_words_ids` is a list of **token sequences**, not of words. This is where BPE tokenization (week 6) becomes important: GPT-2's tokens include the space *before* a word, so `"dog"` at the start of a text and `" dog"` in the middle of a sentence are different tokens, and so are `" Dog"` and `" dogs"`.

In [37]:
for word in ["dog", " dog", " Dog", " dogs", " doggy"]:
    ids = tokenizer(word)["input_ids"]
    print(f"{word!r:10} -> {ids} {tokenizer.convert_ids_to_tokens(ids)}")

'dog'      -> [9703] ['dog']
' dog'     -> [3290] ['Ġdog']
' Dog'     -> [8532] ['ĠDog']
' dogs'    -> [6844] ['Ġdogs']
' doggy'   -> [466, 19970] ['Ġdo', 'ggy']


**3.3 Ban dogs.** Build `banned` for the words `dog`, `dogs`, `Dog` and `Dogs`, each with and without a leading space, and generate a continuation of a prompt about a dog in which none of these token sequences occur.

In [38]:
words = ["dog", "dogs", "Dog", "Dogs"]
banned = [tokenizer(variant)["input_ids"] for word in words for variant in [word, " " + word]]

dogless_ids = generate_ids("I enjoy walking with my cute", max_new_tokens=60, no_repeat_ngram_size=3, bad_words_ids=banned)
print(tokenizer.decode(dogless_ids))

 little sister. I love to play with her and she loves to play. I'm a big fan of her and I love her so much. I've been playing with her for a while now and she's always been my favorite. I think she's a really good girl. I like her a


In [39]:
# ✅ Check your banned words
assert isinstance(banned, list) and len(banned) == 8, "expected 8 token sequences: 4 words, with and without a leading space"
assert [tokenizer.convert_tokens_to_ids("Ġdog")] in banned, "' dog' (with a space) is missing"


def contains(ids, sequence):
    return any(ids[i : i + len(sequence)] == sequence for i in range(len(ids) - len(sequence) + 1))


assert not any(contains(dogless_ids, sequence) for sequence in banned), "a banned token sequence was generated"
print("Looks good ✅")

Looks good ✅


**3.4 Is the text now free of dogs?** Read it carefully. Why is banning words with `bad_words_ids` a weak tool, for example for keeping offensive language out of a model's output?

---

*Answer:*

Usually the model finds a way around the ban: it writes "puppy", "pup", "doggo", "Labrador", "pet", or splits the word into other tokens (`" d" + "og"`, `"DOG"`, `" do" + "gs"`, ...). Every word has many tokenizations and many synonyms, and we only banned eight token sequences.

For safety this is a weak tool for the same reasons: there are endless spellings, synonyms, other languages and paraphrases, and meaning is not tied to particular tokens. A banned-word list can also hurt harmless text ("Scunthorpe problem"). Real systems rely on training the model to behave (fine-tuning and preference alignment, week 7) and on separate classifiers that check inputs and outputs, and use word lists at most as an extra layer.

---

## 4. Looking inside `generate`

`generate` can return much more than the token ids. With `return_dict_in_generate=True` it returns an object with several fields:

- `sequences`: the token ids, as before,
- `scores` (with `output_scores=True`): one tensor of shape `(batch_size, vocab_size)` **per generated token**, with the scores that the next token was chosen from, *after* the logits processors,
- `logits` (with `output_logits=True`): the same, but the raw logits of the model *before* any processing,
- and optionally the attentions and hidden states of every step (`output_attentions`, `output_hidden_states`).

In [40]:
prompt = "The capital of Denmark is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
out = model.generate(**inputs, max_new_tokens=8, return_dict_in_generate=True, output_scores=True)

print(type(out).__name__, list(out.keys()))
print("sequences:", tuple(out.sequences.shape))
print(f"scores: a tuple of {len(out.scores)} tensors of shape {tuple(out.scores[0].shape)}")

GenerateDecoderOnlyOutput ['sequences', 'scores', 'past_key_values']
sequences: (1, 13)
scores: a tuple of 8 tensors of shape (1, 50257)


**4.1 How sure was the model?** Compute the probability that the model gave to each generated token at the moment it was chosen.

Hints:
- `out.scores[t]` holds the scores for the `t`-th new token. A softmax over the vocabulary turns them into probabilities.
- The `t`-th new token is `out.sequences[0, prompt_length + t]`.

In [41]:
prompt_length = inputs["input_ids"].shape[1]
new_tokens = out.sequences[0, prompt_length:]

token_probs = []
for t, token_id in enumerate(new_tokens):
    probs = torch.softmax(out.scores[t][0], dim=-1)
    token_probs.append(probs[token_id].item())
token_probs = torch.tensor(token_probs)

for token_id, p in zip(new_tokens, token_probs):
    print(f"{tokenizer.decode(token_id)!r:>14}  {p:.3f}")
print(f"\nprobability of the whole continuation: {token_probs.prod():.2e}")

        ' the'  0.095
    ' capital'  0.231
         ' of'  0.905
        ' the'  0.338
     ' Danish'  0.111
     ' People'  0.144
          "'s"  0.843
      ' Party'  0.447

probability of the whole continuation: 4.05e-05


Hugging Face has a helper for exactly this, `compute_transition_scores`. It returns the log-probability of every generated token:

In [42]:
# ✅ Check your probabilities against compute_transition_scores
log_probs = model.compute_transition_scores(out.sequences, out.scores, normalize_logits=True)[0]
assert token_probs.shape == (8,), "expected one probability per new token"
assert torch.allclose(token_probs, log_probs.exp().cpu(), atol=1e-4), "your probabilities differ from compute_transition_scores"
print("Looks good ✅")

Looks good ✅


**4.2 Look at the probabilities.** Which tokens was the model sure about and which not? The whole continuation has a tiny probability, even though greedy decoding picked the most likely token at every step. Why is that not a problem in itself, and why does it still mean that the most likely *token* at each step need not lead to the most likely *text*?

---

*Answer:*

The model is typically very sure about tokens that are determined by the context (" Copenhagen" after "The capital of Denmark is" usually gets a high probability, as do the second halves of words split into several tokens and punctuation such as the full stop), and unsure about tokens that start something new: the first word of the next sentence, an adjective, a topic.

The probability of a text is the product of the probabilities of its tokens, and a product of many numbers below 1 quickly becomes tiny. There are astronomically many possible continuations, so even the best one has a small probability; what matters is how it compares with the alternatives (that is why we usually look at log-probabilities and averages per token).

Because the probability of a text is a product, a token with a slightly lower probability now can lead to tokens with much higher probabilities later, and the product can then be larger than along the greedy path. Greedy decoding never looks ahead, so it can miss the most likely text. Beam search (notebook 1) is an attempt to find it.

---

### The scores are *processed* logits

`scores` contains the scores *after* all logits processors. With sampling and top-k (week 6: only the `k` most likely tokens may be sampled), the scores of all other tokens are set to $-\infty$, while the raw `logits` are left untouched.

**4.3 Generate 5 tokens with sampling and `top_k=10`**, returning both the scores and the raw logits, and count at every step how many entries of each are finite (not $-\infty$).

Hint: `torch.isfinite(x).sum()` counts the finite entries of a tensor.

In [43]:
set_seed(0)
out = model.generate(
    **inputs, max_new_tokens=5, do_sample=True, top_k=10,
    return_dict_in_generate=True, output_scores=True, output_logits=True,
)

finite_scores = [int(torch.isfinite(scores[0]).sum()) for scores in out.scores]
finite_logits = [int(torch.isfinite(logits[0]).sum()) for logits in out.logits]

print("finite scores per step:", finite_scores)
print("finite logits per step:", finite_logits)

finite scores per step: [10, 10, 10, 10, 10]
finite logits per step: [50257, 50257, 50257, 50257, 50257]


In [44]:
# ✅ Check your counts
assert len(finite_scores) == 5 and len(finite_logits) == 5, "expected one count per generated token"
assert all(n == 10 for n in finite_scores), "with top_k=10, exactly 10 scores per step should be finite"
assert all(n == model.config.vocab_size for n in finite_logits), "the raw logits should all be finite"
print("Looks good ✅")

Looks good ✅


**4.4 Optional: explore the other outputs.** Generate a few tokens with `output_attentions=True` and `output_hidden_states=True` and look at the structure of `out.attentions` and `out.hidden_states`: how many entries are there, and what are their shapes? Can you explain the shapes with what you know about GPT-2 (12 layers, 12 heads, 768 dimensions) and about how `generate` reuses earlier computations? (We come back to the latter, the *KV cache*, in notebook 2.)

In [47]:
out = model.generate(
    **inputs, max_new_tokens=3, return_dict_in_generate=True, output_attentions=True, output_hidden_states=True
)
print(f"attentions: {len(out.attentions)} steps, each with {len(out.attentions[0])} layers")
for t, step in enumerate(out.attentions):
    print(f"  step {t}: attention of layer 0 has shape {tuple(step[0].shape)}")  # (batch, heads, new positions, all positions)
print(f"hidden states: {len(out.hidden_states)} steps, each with {len(out.hidden_states[0])} tensors (embeddings + 12 layers)")
for t, step in enumerate(out.hidden_states):
    print(f"  step {t}: shape {tuple(step[0].shape)}")  # (batch, new positions, 768)

# At step 0 the whole prompt is processed, so there is one row per prompt token.
# At every later step only the one new token is processed: its query attends to all earlier positions,
# whose keys and values were cached. That is the KV cache.

attentions: 3 steps, each with 0 layers


IndexError: tuple index out of range

## 5. Batched generation

Generating for many prompts one at a time is slow; on a GPU, a batch of prompts takes about as long as a single one. Prompts have different lengths, so they have to be **padded** to the same length, as for classification last week. But for a decoder-only model it matters *on which side* the padding goes.

GPT-2 has no padding token, so we use the EOS token for it, as above. By default the tokenizer pads on the **right**:

In [49]:
tokenizer.pad_token = tokenizer.eos_token
prompts = ["The capital of Denmark is", "My favourite food in the whole wide world is", "Hello"]

tokenizer.padding_side = "right"
batch = tokenizer(prompts, return_tensors="pt", padding=True).to(device)
print(batch["input_ids"])
print(batch["attention_mask"])

outputs = model.generate(**batch, max_new_tokens=15)
for text in tokenizer.batch_decode(outputs, skip_special_tokens=True):
    print(repr(text))

[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


tensor([[  464,  3139,   286, 16490,   318, 50256, 50256, 50256, 50256],
        [ 3666, 12507,  2057,   287,   262,  2187,  3094,   995,   318],
        [15496, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256]],
       device='mps:0')
tensor([[1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 0, 0, 0, 0, 0, 0, 0, 0]], device='mps:0')
'The capital of Denmark is, of course, the capital of the Danish people. It is the capital'
"My favourite food in the whole wide world is the chicken. It's a great way to get a little extra protein and"
"Hello\n\nI'm a big fan of the new game, and I'm"


**5.1 Fix the batch.** Generate the same continuations with padding on the **left** and check that they are the same as when every prompt is generated on its own.

In [50]:
tokenizer.padding_side = "left"
batch = tokenizer(prompts, return_tensors="pt", padding=True).to(device)
outputs = model.generate(**batch, max_new_tokens=15)

new_ids = outputs[:, batch["input_ids"].shape[1]:]
batched = tokenizer.batch_decode(new_ids, skip_special_tokens=True)
one_by_one = [continue_text(prompt, max_new_tokens=15) for prompt in prompts]
for prompt, a, b in zip(prompts, batched, one_by_one):
    print(f"{prompt}|{a!r}\n{' ' * len(prompt)}|{b!r}\n")

The capital of Denmark is|" the capital of the Danish People's Party, which is the party of the"
                         |" the capital of the Danish People's Party, which is the party of the"

My favourite food in the whole wide world is|" the chicken. It's a great way to get a little extra protein and"
                                            |" the chicken. It's a great way to get a little extra protein and"

Hello|", I'm sorry, but I'm not sure if you're aware of"
     |", I'm sorry, but I'm not sure if you're aware of"



In [51]:
# ✅ Check your batch
assert tokenizer.padding_side == "left", "pad on the left"
assert batch["input_ids"].shape[0] == 3 and (batch["input_ids"][:, -1] != tokenizer.pad_token_id).all(), "every prompt should end in the last column"
# Padding can change the numbers very slightly, which may (rarely) flip a near-tie late in the text, so compare the first words
for a, b in zip(batched, one_by_one):
    assert a.split()[:5] == b.split()[:5], f"the batched continuation {a!r} differs from {b!r}"
print("Looks good ✅")

Looks good ✅


**5.2 Why does right padding go wrong for generation**, while it was fine for classifying reviews with DistilBERT last week?

---

*Answer:*

`generate` appends each new token at the **end** of the sequence and predicts it from the output at the **last position**. With right padding, the last position of a short prompt is a padding token, so the model predicts the next token after `"Hello<|endoftext|><|endoftext|>..."`: the real prompt is followed by padding (masked out, but the new tokens still come after it, at the wrong positions), and the continuation is often unrelated to the prompt. With left padding, every prompt ends in the last column, the new tokens follow the real text directly, and the attention mask (and the position ids that `generate` computes from it) make the padding on the left invisible.

DistilBERT's classifier reads the `[CLS]` vector at position **0** and does not generate anything, so padding at the end does not matter there: the attention mask hides it and nothing is appended after it.

---

## 6. MCQ

Answer each question by writing the letter of your choice (A–D) after **Answer:**.

---

### 6.1. Default decoding

You call `model.generate(**inputs, max_new_tokens=50)` on GPT-2 without any other arguments. How is each new token chosen?

A. It is sampled from the full distribution<br>
B. It is the most likely token given the text so far (greedy decoding)<br>
C. Beam search with 5 beams finds the most likely continuation<br>
D. It is sampled from the 50 most likely tokens<br>

**Answer:** **B**

GPT-2's generation config sets neither `do_sample` nor `num_beams`, so `generate` uses greedy decoding: the argmax at every step. That is why the output is deterministic and why it tends to repeat itself.

---

### 6.2. Stopping

Which of the following does **not** stop generation?

A. The model generates the token given as `eos_token_id`<br>
B. `max_new_tokens` tokens have been generated<br>
C. The model generates the same token twice in a row<br>
D. One of the `stop_strings` appears in the output<br>

**Answer:** **C**

Generation stops at the EOS token (A), at the length limit (B) or at a stop string (D). A repeated token is no reason to stop; that is exactly how greedy loops can run until `max_new_tokens`.

---

### 6.3. Scores

You call `generate` with `do_sample=True, top_k=10, output_scores=True, output_logits=True, return_dict_in_generate=True`. What is the difference between `scores` and `logits`?

A. There is none; both are the raw output of the model<br>
B. `scores` are probabilities, `logits` are log-probabilities<br>
C. `scores` are after the logits processors (here all but 10 entries are $-\infty$), `logits` are the raw outputs of the model<br>
D. `scores` contain one tensor per layer, `logits` one per generated token<br>

**Answer:** **C**

`scores` are what the next token was actually chosen from, after processors such as temperature, top-k, repetition penalties or banned words. `logits` are the model's unmodified outputs. Both contain one tensor per generated token, and neither is normalised to probabilities.

---

### 6.4. Banned words

You want to stop GPT-2 from writing the word *cat* and pass `bad_words_ids=[tokenizer("cat")["input_ids"]]`. Why can the model still write "My cat is"?

A. `bad_words_ids` only works with beam search<br>
B. Inside a sentence, *cat* is the token `" cat"` with a leading space, which is a different token from `"cat"`<br>
C. `bad_words_ids` only lowers the probability of the token, it does not forbid it<br>
D. GPT-2's tokenizer lowercases all text<br>

**Answer:** **B**

GPT-2's BPE tokens include the preceding space, so the same word has (at least) two tokens: one at the start of a text and one after a space, plus capitalised and plural variants. `bad_words_ids` forbids exactly the token sequences you give it, and it does forbid them completely (C is wrong).

---

### 6.5. Padding for generation

Why are prompts padded on the **left** when a decoder-only model generates for a batch of prompts?

A. So that the last position of every row is a real token, and the new tokens are appended directly after the prompt<br>
B. Because the attention mask only works for padding on the left<br>
C. Because GPT-2 reads text from right to left<br>
D. Left padding makes the batch shorter<br>

**Answer:** **A**

`generate` predicts the next token from the last position and appends it at the end. With left padding every prompt ends in the last column. The attention mask works on either side (B), and the amount of padding is the same on both sides (D).